In [1]:
from deepanc.common.utils import loading_paths_from_MAT, nmse, apply_impulse_response
from deepanc.inference.antinoise_generator import Deep_ANC_controller
import torch
import numpy as np
import math
import soundfile as sf
from construct_dataset_from_freesound.resample import resampler

In [2]:
from pyprojroot import here
PROJECT_ROOT = here()
MODEL_DIR = PROJECT_ROOT / "models"
PATHS_DIR = PROJECT_ROOT / "rirs"
NOISE_DIR = PROJECT_ROOT / "NOISEX92"
FIG_DIR = PROJECT_ROOT / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)
print(MODEL_DIR)
print(PATHS_DIR)
print(NOISE_DIR)


/home/rkobayashi/anc/deepanc
/home/rkobayashi/anc/deepanc/models
/home/rkobayashi/anc/deepanc/rirs
/home/rkobayashi/anc/deepanc/NOISEX92


In [16]:
fs = 16000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
delay = 1
rir = [f"rt60_{i}s" for i in [0.15, 0.175, 0.2, 0.225, 0.25]][2]
eta = torch.tensor([math.sqrt(0.1), math.sqrt(1), math.sqrt(10), torch.finfo(torch.float32).max], dtype=torch.float32)[0]
model_name = f"synthesized_delay_{delay}.pth"

In [17]:
Pri_path, Sec_path = torch.tensor(loading_paths_from_MAT(PATHS_DIR, rir, "Primary_path.mat", "Secondary_path.mat", )).unsqueeze(1)

sound_names = [ sound_name.name + "/signal.wav" for sound_name in NOISE_DIR.iterdir() if sound_name.is_dir()]

for sound_index in range(len(sound_names)):
    sound_name = sound_names[sound_index]
    print(sound_name)
    waveform, sample_rate = sf.read(NOISE_DIR/sound_name, dtype="float32", )
    waveform = waveform[:fs*10]
    waveform = torch.from_numpy(resampler(waveform, sample_rate, fs))

    Dis, Ref = apply_impulse_response(waveform, Pri_path), waveform
    Time = np.arange(Dis.shape[1]) / fs

    controller = Deep_ANC_controller(MODEL_DIR/model_name, device, fs, delay, eta, Sec_path)
    Error = controller.noise_cancellation(Dis, Ref)

    print(f"  NMSE: {nmse(Error.detach(), Dis.detach())}")

    # Dis, Error = Dis.tolist()[0], Error.tolist()[0]

m109/signal.wav
  NMSE: 0.08265040069818497
destroyerengine/signal.wav
  NMSE: -2.454871892929077
hfchannel/signal.wav
  NMSE: -4.287335395812988
buccaneer1/signal.wav
  NMSE: -0.825079083442688
factory2/signal.wav
  NMSE: 0.2674919664859772
f16/signal.wav
  NMSE: 0.19459368288516998
babble/signal.wav
  NMSE: -0.5478801727294922
white/signal.wav
  NMSE: 2.6619510650634766
pink/signal.wav
  NMSE: 1.4716379642486572
volvo/signal.wav
  NMSE: 0.44542860984802246
leopard/signal.wav
  NMSE: -0.4629113972187042
destroyerops/signal.wav
  NMSE: -0.26519525051116943
buccaneer2/signal.wav
  NMSE: -0.13452264666557312
machinegun/signal.wav
  NMSE: 0.5175330638885498
factory1/signal.wav
  NMSE: 0.5846831798553467


In [ ]:
# import matplotlib.pyplot as plt
# plt.title('The DeepANC Algorithm')
# plt.plot(Time, Dis, color='blue', label='ANC off')
# plt.plot(Time, Error, color='orange', label='ANC on')
# plt.ylabel('Magnitude')
# plt.xlabel('Time (seconds)')
# plt.legend()
# plt.grid()
# plt.savefig(FIG_DIR/'DeepANC.png', dpi=600, bbox_inches='tight', pad_inches=0)
# plt.show()